# Error 0.5 AI Model Experiment - Demo

Этот ноутбук демонстрирует работу с теорией «ошибки 0.5» для анализа LLM.

In [ ]:
# Импорт библиотек
import sys
sys.path.insert(0, '..')

from src import ExperimentRunner, load_config, MetricsCalculator
from src.utils import normalize_response, calculate_category_distribution
import pandas as pd
import matplotlib.pyplot as plt

## Загрузка конфигурации

In [ ]:
# Загрузить конфигурацию
config = load_config('../config.yaml')
print("Configuration loaded:")
print(f"  Provider: {config['llm']['provider']}")
print(f"  Temperature: {config['generation']['temperature']}")
print(f"  Repetitions: {config['experiment']['default_repetitions']}")

## Пример нормализации ответов

In [ ]:
# Тестирование нормализации
test_responses = [
    "Yes, that is correct.",
    "I don't know the answer.",
    "The capital of France is Paris.",
    "Maybe, but I'm not sure.",
    "No, that's incorrect.",
    "42 is the answer to everything.",
]

print("Response Normalization Demo:\n")
for response in test_responses:
    category = normalize_response(response, method='keyword')
    print(f"'{response[:40]}...' -> {category}")

## Расчёт метрик вручную

In [ ]:
# Создать калькулятор метрик
calculator = MetricsCalculator(
    quality_threshold=0.8,
    p_gain=1.0,
    b_value=1.0,
    p_loss=0.5,
    l_value=0.5
)

# Пример категорий из эксперимента
sample_categories = [
    'informative', 'informative', 'informative',
    'uncertain', 'informative', 'affirmative',
    'informative', 'ambiguous', 'informative',
    'informative'
]

# Рассчитать все метрики
metrics = calculator.calculate_all_metrics(sample_categories)

print("Metrics for sample responses:\n")
print(f"  Frequencies: {metrics['frequencies']}")
print(f"  K_rep (Repetition): {metrics['k_rep']:.4f}")
print(f"  K_div (Diversity): {metrics['k_div']:.4f}")
print(f"  Entropy (H): {metrics['entropy']:.4f} bits")
print(f"  D_lack (Data Lack): {metrics['d_lack']:.4f}")
print(f"  Ω_0.5 (Useful Error): {metrics['omega_05']:.4f}")

## Визуализация распределения

In [ ]:
# Визуализация частот категорий
freq = metrics['frequencies']

plt.figure(figsize=(10, 6))
plt.bar(freq.keys(), freq.values(), color='steelblue', alpha=0.7)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Response Category Distribution', fontsize=14)
plt.axhline(y=1/len(freq), color='r', linestyle='--', label='Uniform distribution')
plt.legend()
plt.tight_layout()
plt.show()

# Круговая диаграмма
plt.figure(figsize=(8, 8))
plt.pie(freq.values(), labels=freq.keys(), autopct='%1.1f%%', startangle=90)
plt.title('Category Proportions', fontsize=14)
plt.tight_layout()
plt.show()

## Сравнение двух распределений

In [ ]:
# Два распределения для сравнения (например, при разных температурах)
dist_low_temp = {'informative': 0.9, 'uncertain': 0.1}
dist_high_temp = {'informative': 0.5, 'uncertain': 0.2, 'ambiguous': 0.2, 'other': 0.1}

# Сравнить
comparison = calculator.compare_distributions(dist_low_temp, dist_high_temp)

print("Comparison: Low Temp → High Temp\n")
print("Category Changes:")
for cat, data in comparison['category_changes'].items():
    print(f"  {cat}: {data['before']:.2f} → {data['after']:.2f} (Δ={data['delta']:+.2f})")

print(f"\nK_rep change: {comparison['k_rep_change']:+.4f}")
print(f"Entropy change: {comparison['entropy_change']:+.4f} bits")
print(f"KL divergence: {comparison['kl_divergence']:.4f}")

## Запуск полного эксперимента (симуляция)

In [ ]:
# Для реального эксперимента раскомментируйте и настройте:
# 
# runner = ExperimentRunner(config)
# 
# prompts = [
#     "What is 2+2?",
#     "Explain quantum computing in one sentence",
#     "Is the sky blue?"
# ]
# 
# results = runner.run_experiment(
#     prompts=prompts,
#     n_repetitions=10,  # Начните с малого для теста
#     temperature=0.7,
# )
# 
# print(results['summary'])

print("Для запуска реального эксперимента:")
print("1. Установите OPENAI_API_KEY или используйте локальную модель")
print("2. Раскомментируйте код выше")
print("3. Запустите ячейку")

## Интерпретация результатов

In [ ]:
def interpret_metrics(k_rep, entropy, d_lack, omega_05):
    """Интерпретировать метрики эксперимента."""
    
    print("=" * 50)
    print("INTERPRETATION")
    print("=" * 50)
    
    # K_rep
    if k_rep > 0.8:
        print(f"✓ K_rep={k_rep:.2f}: Очень высокая повторяемость")
        print("  → Модель стабильна, подходит для фактологических задач")
    elif k_rep > 0.5:
        print(f"✓ K_rep={k_rep:.2f}: Хороший баланс")
        print("  → Оптимально для большинства задач")
    else:
        print(f"⚠ K_rep={k_rep:.2f}: Низкая повторяемость")
        print("  → Модель непредсказуема, проверьте температуру")
    
    # Entropy
    if entropy > 2.0:
        print(f"\n⚠ H={entropy:.2f} bits: Высокая энтропия")
        print("  → Много разнообразных ответов")
    else:
        print(f"\n✓ H={entropy:.2f} bits: Низкая/средняя энтропия")
        print("  → Ответы предсказуемы")
    
    # D_lack
    if d_lack > 0.3:
        print(f"\n⚠ D_lack={d_lack:.2f}: Высокий индекс нехватки данных")
        print("  → Модель часто не уверена в ответах")
        print("  → Возможно, требуется больше контекста/RAG")
    else:
        print(f"\n✓ D_lack={d_lack:.2f}: Низкий индекс нехватки")
        print("  → Модель уверена в ответах")
    
    # Omega_0.5
    if omega_05 > 0.3:
        print(f"\n✓ Ω_0.5={omega_05:.2f}: Положительная полезная ошибка")
        print("  → Разнообразие приносит пользу")
    elif omega_05 > -0.3:
        print(f"\n~ Ω_0.5={omega_05:.2f}: Баланс")
        print("  → Польза и вред примерно равны")
    else:
        print(f"\n⚠ Ω_0.5={omega_05:.2f}: Отрицательная полезная ошибка")
        print("  → Ошибки перевешивают преимущества")
    
    print("=" * 50)

# Пример интерпретации
interpret_metrics(
    k_rep=0.65,
    entropy=1.5,
    d_lack=0.15,
    omega_05=0.45
)